# 02 · Silver · `faturas`
### V-Energy Grid 360 — Arquitetura Medalhão

Lê `v_energy.bronze.faturas` (`faturas.csv`, separador `;`; tudo como texto) e grava `v_energy.silver.faturas` com os dados limpos, tipados e confiáveis para a Gold.

**Fonte das regras:** `exploracao_faturas.ipynb` e a seção `faturas` do `docs/data_quality_log.md` (itens F1 a F5). As decisões foram tomadas pela responsável pela tabela, com a autonomia combinada com o grupo, e estão registradas no log com as alternativas consideradas.

| # | Tratamento | Linhas afetadas | Regra |
|---|---|---|---|
| 1 | Linhas de cabeçalho repetido (F1) | 2 | remover, **antes** da deduplicação |
| 2 | Cópias exatas | 0 | `dropDuplicates(["id_fatura"])` (a exploração não encontrou nenhuma; fica como proteção) |
| 3 | Faturas órfãs: UC inexistente no cadastro (F2) | 32.285 | remover |
| 4 | `valor_total` escrito como moeda, `R$ 9,9` e `R$ 9.9,9` (F3) | 196.622 na Bronze (inclui as órfãs) | tirar `R$` e o ponto de milhar, vírgula vira ponto, `decimal(12,2)` |
| 5 | `data_emissao` em 4 formatos (F4) | 183.361 fora do ISO (inclui as órfãs) | converter cada padrão com o seu formato e gravar como `date` |
| 6 | `data_pagamento` ausente em 2 formas: nulo real e `9999-12-31` (F5) | 175.168 na Bronze (inclui as órfãs) | `9999-12-31` vira nulo; flag `em_aberto` |

**Ordem das operações (importa):**
1. Remover o cabeçalho repetido e só depois deduplicar (se deduplicar primeiro, uma das 2 linhas de cabeçalho sobra).
2. Remover as órfãs.
3. Converter os valores e as datas, e só então derivar `em_aberto`.

**Resultado esperado:** 2.152.336 linhas, com `(id_uc, mes_referencia)` único, e exatamente uma fatura para cada leitura de `silver.leituras`.

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row
from datetime import datetime

tabela_bronze = f"{bronze_schema}.faturas"
tabela_silver = f"{silver_schema}.faturas"

print(f"origem:  {tabela_bronze}")
print(f"destino: {tabela_silver}")

origem:  v_energy.bronze.faturas
destino: v_energy.silver.faturas


## 2. Funções de checagem de qualidade

Mesmas funções da aula `02_Bronze_to_Silver`, para que as três Silvers do grupo gravem em `silver.dq_log` com as mesmas colunas (`table_name`, `check_name`, `total_rows`, `failed_rows`, `passed`, `checked_at`).

- `dq_check(table_name, check_name, df, condition)`: `condition` é a condição **esperada** (o que deveria ser verdade em todas as linhas). Falham as linhas que a violam.
- `dq_check_unique(table_name, check_name, df, key_cols)`: passa se a chave não se repete.
- `dq_check_contagem(table_name, check_name, df, esperado)`: passa só se o número de linhas for exatamente o esperado.

As funções só acumulam os resultados na lista `dq_results`. Depois das checagens, o notebook faz duas coisas, nesta ordem: **grava o resultado em `silver.dq_log`** (sempre, inclusive quando há falha, porque o registro da falha também é útil) e **só então decide se grava a Silver** (só se todas as checagens passarem).

**Cuidado com nulos:** linhas em que a condição resulta em nulo não contam como falha (o filtro `~condition` descarta nulos). Quando o nulo também for um problema, escreva um `dq_check` separado com `isNotNull()`.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def n_br(n):
    return f"{n:,}".replace(",", ".")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=diferenca, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

## 3. Leitura da Bronze

Leitura da tabela Bronze, sem transformação nenhuma. Duas conferências antes de começar:
- A contagem deve ser **2.184.623** linhas (2.184.621 de dados e 2 de cabeçalho repetido).
- Todas as colunas devem aparecer como `string`: é a regra da Bronze. As conversões de tipo só acontecem aqui na Silver.

O DataFrame `bronze` fica guardado para a comparação Bronze × Silver no fim do notebook.

In [0]:
bronze = spark.table(tabela_bronze)

n_bronze = bronze.count()
print(f"Linhas na Bronze: {n_br(n_bronze)}")  # esperado: 2.184.623

bronze.printSchema()
display(bronze.limit(5))

Linhas na Bronze: 2.184.623
root
 |-- id_fatura: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- data_emissao: string (nullable = true)
 |-- valor_energia: string (nullable = true)
 |-- tributos: string (nullable = true)
 |-- bandeira_tarifaria: string (nullable = true)
 |-- valor_total: string (nullable = true)
 |-- data_vencimento: string (nullable = true)
 |-- data_pagamento: string (nullable = true)
 |-- status: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,bandeira_tarifaria,valor_total,data_vencimento,data_pagamento,status,source_file,ingestion_timestamp
FT001417237,UC129634,2023-02,2023-02-05,175.72,58.57,Verde,234.29,2023-02-17,2023-02-20,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
FT001417238,UC129634,2023-03,2023-03-13,191.99,64.00,Verde,255.99,2023-03-25,2023-03-23,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
FT001417239,UC129634,2023-04,2023-04-15,142.77,47.59,Verde,190.36,2023-04-27,2023-05-20,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
FT001417240,UC129634,2023-05,05-28-2023,142.21,47.40,Verde,189.61,2023-06-09,2023-06-26,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z
FT001417241,UC129634,2023-06,2023-06-03,104.67,34.89,Verde,139.56,2023-06-15,2023-07-08,PAGA,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv,2026-10-19T20:11:35.176Z


## 4. Linhas de cabeçalho repetido (log F1)

**Decisão:** remover as linhas em que `id_fatura` é igual a `"id_fatura"`.

**Evidência:** a exploração encontrou 2 linhas em que o valor de cada coluna é o nome da própria coluna (`data_emissao = 'data_emissao'` e assim por diante), ambas vindas do `faturas.csv`. É um nulo estrutural: o registro é inutilizável.

**Por que antes da deduplicação:** as 2 linhas são idênticas entre si, então uma deduplicação por `id_fatura` deixaria uma delas na tabela.

**Alternativa considerada:** corrigir na Bronze, o que evitaria o tratamento em cada Silver. Depende de verificar se o cabeçalho está repetido no arquivo de origem ou se é efeito da leitura.

**Conferência:** 2 linhas removidas; restam 2.184.621.

In [0]:
# Tratamento 1 · cabeçalho repetido (esperado: 2 removidas, 2.184.621 restantes)
# coalesce: se id_fatura fosse nulo, a linha NÃO seria descartada aqui (ela falharia nas checagens, que é o certo)
e_cabecalho = F.coalesce(F.col("id_fatura") == "id_fatura", F.lit(False))
df_sem_cab = bronze.filter(~e_cabecalho)

n_sem_cab = df_sem_cab.count()
print(f"Cabeçalhos removidos: {n_br(n_bronze - n_sem_cab)} | antes: {n_br(n_bronze)} | depois: {n_br(n_sem_cab)}")

Cabeçalhos removidos: 2 | antes: 2.184.623 | depois: 2.184.621


## 5. Cópias exatas

**Decisão:** `dropDuplicates(["id_fatura"])`, como proteção.

**Evidência:** a exploração não encontrou cópias: 2.184.621 linhas, 2.184.621 `id_fatura` distintos e o mesmo número de linhas distintas sem as colunas de auditoria. Este passo é só uma garantia caso uma carga futura traga repetições.

**Conferência:** 0 linhas removidas.

In [0]:
# Tratamento 2 · cópias exatas (esperado: 0 removidas)
df_dedup = df_sem_cab.dropDuplicates(["id_fatura"])

n_dedup = df_dedup.count()
print(f"Cópias removidas: {n_br(n_sem_cab - n_dedup)} | antes: {n_br(n_sem_cab)} | depois: {n_br(n_dedup)}")

Cópias removidas: 0 | antes: 2.184.621 | depois: 2.184.621


## 6. Faturas órfãs: UC inexistente no cadastro (log F2)

**Decisão:** remover. Alternativa considerada: gravar `df_orfas` em uma tabela de quarentena. Descartada porque a Bronze já guarda as linhas e o padrão indica registros espúrios. Se a V-Energy confirmar que são faturas reais, a quarentena passa a fazer sentido.

**Evidência:** 32.285 faturas (1,5%) têm as três condições ao mesmo tempo: `id_fatura` fora do padrão `FT` + 9 dígitos (prefixo `FTX`), `id_uc` com 7 dígitos e UC inexistente em `unidades_consumidoras`. As outras 2.152.336 estão todas no padrão e com UC existente. As 32.285 órfãs correspondem a 31.705 UCs distintas (cerca de uma fatura por UC), nenhuma com leitura nem ocorrência de rede em outra tabela, distribuídas de forma uniforme no tempo. O padrão é compatível com registros espúrios, e não com clientes reais fora do cadastro (hipótese: a confirmação cabe à V-Energy). Somam R$ 7,64 milhões em `valor_energia` (1,5% do total).

**Como:** join `left_semi` (mantém) e `left_anti` (órfãs) contra os ids **distintos** de `unidades_consumidoras`, descartando a linha de cabeçalho repetido que essa tabela também tem (`id_uc = 'id_uc'`). O critério é a existência da UC, que é a integridade referencial exigida pelo case. O formato dos ids é conferido nas checagens.

**Impacto na Gold:** a receita total fica ~1,5% menor sem as órfãs.

**Conferência:** 32.285 linhas removidas; restam 2.152.336, com `(id_uc, mes_referencia)` único.

In [0]:
# Tratamento 3 · órfãs: UC inexistente no cadastro (esperado: 32.285 removidas, 2.152.336 restantes)
ids_ucs = (
    spark.table(f"{bronze_schema}.unidades_consumidoras")
    .filter(~F.coalesce(F.col("id_uc") == "id_uc", F.lit(False)))  # unidades_consumidoras também tem uma linha de cabeçalho repetido
    .select("id_uc").distinct()
)

df_orfas = df_dedup.join(ids_ucs, "id_uc", "left_anti")
df_validas = df_dedup.join(ids_ucs, "id_uc", "left_semi")

n_validas = df_validas.count()
print(f"Órfãs removidas: {n_br(n_dedup - n_validas)} | antes: {n_br(n_dedup)} | depois: {n_br(n_validas)}")
display(df_orfas.select("id_fatura", "id_uc", "mes_referencia", "valor_energia", "valor_total").limit(5))

Órfãs removidas: 32.285 | antes: 2.184.621 | depois: 2.152.336


id_fatura,id_uc,mes_referencia,valor_energia,valor_total
FTX0000021,UC9404512,2025-11,162.28,216.37
FTX0000024,UC9711370,2024-11,385.74,514.32
FTX0000057,UC9600703,2023-03,102.61,136.81
FTX0000083,UC9985148,2026-01,66.00,88.00
FTX0000103,UC9896403,2025-01,100.97,134.63


## 7. Valores monetários: `valor_total` como moeda (log F3)

**Decisão:** normalizar o texto de `valor_total` antes do cast: tirar `R$` e o ponto de milhar e trocar a vírgula decimal por ponto. Depois, `decimal(12,2)` nas três colunas de valor (`valor_energia`, `tributos` e `valor_total`).

**Evidência:** `valor_energia` e `tributos` têm um único formato (`9.9`, ponto decimal). `valor_total` tem três: `9.9` (`102.57`), `R$ 9,9` (`R$ 100,64`) e `R$ 9.9,9` (`R$ 1.234,56`, com ponto de milhar). Na Bronze, 196.622 faturas (9,0%) têm o total como moeda. Um `cast` simples as anularia, e a receita sairia ~9% menor do que a real. Com a normalização, todos os totais das faturas válidas convertem e `valor_total = valor_energia + tributos` em 100% (diferença zero).

**Detalhes:**
- Só os valores que começam com `R$` passam pela normalização. Nos demais, o ponto já é o separador decimal, e tirá-lo destruiria o valor.
- O símbolo pode vir seguido de espaço comum ou de espaço sem quebra (`\u00a0`), e a expressão cobre os dois.
- `try_cast` em todas as conversões: o `cast` comum dá erro com valores inválidos.
- As colunas `*_txt` guardam o texto original e servem a uma checagem: se existisse algum valor com mais de 2 casas decimais, o cast para `decimal(12,2)` arredondaria sem avisar. Elas não vão para a Silver.

**Alternativa considerada:** recalcular `valor_total` como `valor_energia + tributos`. Descartada por ser desnecessária: a normalização converte todos os totais e eles batem com a soma.

**Conferência:** nenhum valor nulo depois do cast.

In [0]:
# Tratamento 4 · valores monetários (esperado: 0 nulos nas três colunas depois do cast)
vt = F.col("valor_total")
total_normalizado = F.when(
    vt.startswith("R$"),
    F.regexp_replace(F.regexp_replace(F.regexp_replace(vt, r"R\$[\s\u00a0]*", ""), r"\.", ""), ",", "."),
).otherwise(vt)

df_valores = (
    df_validas
    .withColumn("valor_energia_txt", F.col("valor_energia"))   # auxiliares: texto original, para uma checagem
    .withColumn("tributos_txt", F.col("tributos"))
    .withColumn("valor_total_txt", total_normalizado)
    .withColumn("valor_energia", F.expr("try_cast(valor_energia_txt as decimal(12,2))"))
    .withColumn("tributos", F.expr("try_cast(tributos_txt as decimal(12,2))"))
    .withColumn("valor_total", F.expr("try_cast(valor_total_txt as decimal(12,2))"))
)

# informativo: quantas faturas válidas tinham o total como moeda (na Bronze, com as órfãs, eram 196.622)
print("totais escritos como moeda (R$):", n_br(df_validas.filter(F.col("valor_total").startswith("R$")).count()))
for c in ["valor_energia", "tributos", "valor_total"]:
    print(f"{c} nulo após o cast:", n_br(df_valores.filter(F.col(c).isNull()).count()))

totais escritos como moeda (R$): 193.710
valor_energia nulo após o cast: 0
tributos nulo após o cast: 0
valor_total nulo após o cast: 0


## 8. Datas: `data_emissao` em quatro formatos e `data_pagamento` com sentinela (log F4 e F5)

### `data_emissao` (F4)
**Decisão:** converter cada padrão com o seu formato e gravar como `date`.

| Padrão | Formato | Linhas (Bronze) |
|---|---|---|
| `9999-99-99` | `yyyy-MM-dd` | 2.001.260 |
| `99/99/9999` | `dd/MM/yyyy` | 61.291 |
| `99.99.9999` | `dd.MM.yyyy` | 61.252 |
| `99-99-9999` | `MM-dd-yyyy` (**mês primeiro**) | 60.818 |

**Evidência da ordem dia/mês:** o vencimento é sempre a emissão + 12 dias. Com os formatos acima, isso vale em 100% das linhas e nenhuma deixa de converter. Os formatos alternativos falham (por exemplo, `dd-MM-yyyy` não converte 38.615 linhas). É o mesmo mapeamento de `leituras`: hífen é mês primeiro.

**Risco evitado:** aplicar um formato único converteria datas para o mês errado, sem dar erro.

### `data_vencimento`
Formato único `yyyy-MM-dd`, convertida para `date`.

### `data_pagamento` (F5)
**Decisão:** converter `9999-12-31` em nulo real. Os nulos reais continuam nulos.

**Evidência:** a ausência de pagamento aparece de duas formas: nulo real (113.835) e a sentinela `9999-12-31` (61.333), 175.168 faturas em aberto (8,0%) na Bronze. A sentinela só aparece em faturas `EM ABERTO`, com distribuição regular (~1.275 por mês). Todas as 2.009.453 `PAGA` têm data real. Se a sentinela ficasse, contas como "dias entre vencimento e pagamento" usariam o ano 9999.

**Alternativa considerada:** manter a sentinela. Descartada porque distorceria qualquer média ou diferença de datas.

**Conferência:** nenhuma `data_emissao` nem `data_vencimento` nula depois da conversão, e nenhuma `data_pagamento` igual à sentinela.

In [0]:
# Tratamento 5 e 6 · datas (esperado: 0 emissões e 0 vencimentos nulos; 0 pagamentos na sentinela)
formatos = {
    "9999-99-99": "yyyy-MM-dd",
    "99-99-9999": "MM-dd-yyyy",   # hífen = MÊS primeiro
    "99.99.9999": "dd.MM.yyyy",
    "99/99/9999": "dd/MM/yyyy",
}

emissao_convertida = None
for padrao, fmt in formatos.items():
    cond = F.col("padrao_data") == padrao
    valor = F.expr(f"to_date(try_to_timestamp(data_emissao, '{fmt}'))")
    emissao_convertida = F.when(cond, valor) if emissao_convertida is None else emissao_convertida.when(cond, valor)

SENTINELA = "9999-12-31"

df_datas = (
    df_valores
    .withColumn("padrao_data", F.regexp_replace("data_emissao", "[0-9]", "9"))
    .withColumn("data_emissao", emissao_convertida)   # lê o texto original e substitui pela data
    .withColumn("data_vencimento", F.expr("to_date(try_to_timestamp(data_vencimento, 'yyyy-MM-dd'))"))
    .withColumn(
        "data_pagamento",
        F.when(F.col("data_pagamento") == SENTINELA, F.lit(None).cast("date"))
         .otherwise(F.expr("to_date(try_to_timestamp(data_pagamento, 'yyyy-MM-dd'))")),
    )
)

print("data_emissao nula após a conversão:   ", n_br(df_datas.filter("data_emissao IS NULL").count()))
print("data_vencimento nula após a conversão:", n_br(df_datas.filter("data_vencimento IS NULL").count()))
print("data_pagamento na sentinela:          ", n_br(df_datas.filter(F.year("data_pagamento") == 9999).count()))

# informativo: como a ausência de pagamento estava escrita nas faturas válidas
# (na Bronze, com as órfãs: 113.835 nulos reais e 61.333 sentinelas)
print("pagamento ausente, nulo real:", n_br(df_validas.filter(F.col("data_pagamento").isNull()).count()),
      "| sentinela:", n_br(df_validas.filter(F.col("data_pagamento") == SENTINELA).count()))

data_emissao nula após a conversão:    0
data_vencimento nula após a conversão: 0
data_pagamento na sentinela:           0
pagamento ausente, nulo real: 112.159 | sentinela: 60.394


## 9. Padronização e flag `em_aberto` (log F5)

**Decisão:** derivar a flag `em_aberto = (status == 'EM ABERTO')`.

**Evidência:** `status` só tem `PAGA` e `EM ABERTO`, sem variações de grafia. Todas as faturas `PAGA` têm data de pagamento real, e todas as `EM ABERTO` têm data nula (depois de converter a sentinela). A flag é equivalente à ausência da data, mas é mais clara para quem consome a tabela.

**Padronização:** `status` em `upper(trim)` (já vinha assim, o tratamento só garante). `bandeira_tarifaria` só com `trim`: a exploração não encontrou variações de grafia, e mudar a caixa alteraria valores que a Gold e o portal exibem (`Verde`, `Amarela`, `Vermelha P1`).

**Impacto na Gold:** a lista de "faturas em aberto" (tela do atendente) deve usar o `status` ou a flag `em_aberto`, e não a presença da data.

**Conferência:** `em_aberto` coerente com a data de pagamento (checagem na seção 11).

In [0]:
# Padronização e flag em_aberto
df_final = (
    df_datas
    .withColumn("status", F.upper(F.trim("status")))
    .withColumn("bandeira_tarifaria", F.trim("bandeira_tarifaria"))
    .withColumn("em_aberto", F.col("status") == "EM ABERTO")
)

df_final.groupBy("status", "em_aberto").count().orderBy("status").show()

+---------+---------+-------+
|   status|em_aberto|  count|
+---------+---------+-------+
|EM ABERTO|     true| 172553|
|     PAGA|    false|1979783|
+---------+---------+-------+



## 10. Seleção final das colunas

Colunas auxiliares (`valor_energia_txt`, `tributos_txt`, `valor_total_txt` e `padrao_data`) ficam de fora. Fica uma flag para a Gold: `em_aberto`.

In [0]:
# Seleção final das colunas da Silver (descarta as colunas auxiliares)
df_silver = df_final.select(
    "id_fatura", "id_uc", "mes_referencia",
    "data_emissao", "data_vencimento", "data_pagamento",
    "valor_energia", "tributos", "valor_total",
    "bandeira_tarifaria", "status", "em_aberto",
    "source_file",
)
df_silver.printSchema()

root
 |-- id_fatura: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- data_emissao: date (nullable = true)
 |-- data_vencimento: date (nullable = true)
 |-- data_pagamento: date (nullable = true)
 |-- valor_energia: decimal(12,2) (nullable = true)
 |-- tributos: decimal(12,2) (nullable = true)
 |-- valor_total: decimal(12,2) (nullable = true)
 |-- bandeira_tarifaria: string (nullable = true)
 |-- status: string (nullable = true)
 |-- em_aberto: boolean (nullable = true)
 |-- source_file: string (nullable = true)



## 11. Checagens de qualidade

Rodam sobre o resultado, **antes** de gravar. Cada uma protege uma decisão acima:

| Checagem | Protege |
|---|---|
| Total de linhas = 2.152.336 | cabeçalho e órfãs removidos na quantidade esperada (vale para esta extração) |
| `id_fatura` e `(id_uc, mes_referencia)` únicos | cópias e órfãs |
| Colunas-chave não nulas | conversões de número e de data |
| `id_fatura` = `FT` + 9 dígitos, `id_uc` = `UC` + 6 dígitos, `mes_referencia` = `yyyy-MM` | órfãs e formato das chaves |
| `status` só `PAGA` ou `EM ABERTO` | domínio da categoria |
| Valores >= 0 e `valor_total = valor_energia + tributos` | normalização da moeda |
| Valores de origem com no máximo 2 casas decimais | o cast para `decimal(12,2)` não arredondou nada |
| Emissão <= vencimento e vencimento = emissão + 12 dias | ordem dia/mês das datas |
| `em_aberto` = ausência de `data_pagamento` | sentinela e flag |
| Pagamento >= emissão e nenhum pagamento no ano 9999 | sentinela convertida |
| Toda fatura tem leitura no mesmo mês, e toda leitura tem fatura (`silver.leituras`) | integridade referencial entre medição e faturamento |

**Observações:**
- A checagem de contagem total é uma proteção desta extração única: se a Silver for reexecutada com dados novos, o valor esperado precisa ser atualizado.
- As duas últimas checagens leem `silver.leituras`, então a Silver de leituras precisa estar gravada.
- As checagens marcadas como **nova** não foram feitas na exploração. Se alguma falhar, investigue antes de mudar a checagem.

In [0]:
# Checagens de qualidade antes de gravar (condição = o que deve ser verdade em todas as linhas)
t = "silver.faturas"
dq_results.clear()  # se esta célula for reexecutada, não acumula resultados antigos

# Volume e chaves
dq_check_contagem(t, "total de linhas = 2.152.336", df_silver, 2_152_336)
dq_check_unique(t, "id_fatura único", df_silver, ["id_fatura"])
dq_check_unique(t, "(id_uc, mes_referencia) único", df_silver, ["id_uc", "mes_referencia"])

# Nulos (data_pagamento pode ser nula: é nulo de negócio, fatura em aberto)
for coluna in ["id_fatura", "id_uc", "mes_referencia", "data_emissao", "data_vencimento",
               "valor_energia", "tributos", "valor_total", "bandeira_tarifaria", "status"]:
    dq_check(t, f"{coluna} não nulo", df_silver, F.col(coluna).isNotNull())   # bandeira_tarifaria: nova

# Formato das chaves e domínios
dq_check(t, "id_fatura no padrão FT + 9 dígitos", df_silver, F.col("id_fatura").rlike("^FT[0-9]{9}$"))
dq_check(t, "id_uc no padrão UC + 6 dígitos", df_silver, F.col("id_uc").rlike("^UC[0-9]{6}$"))
dq_check(t, "mes_referencia no formato yyyy-MM", df_silver, F.col("mes_referencia").rlike("^[0-9]{4}-[0-9]{2}$"))
dq_check(t, "status só PAGA ou EM ABERTO", df_silver,
         F.coalesce(F.col("status").isin("PAGA", "EM ABERTO"), F.lit(False)))  # nulo também falha

# Valores
dq_check(t, "valor_energia >= 0", df_silver, F.col("valor_energia") >= 0)       # nova
dq_check(t, "tributos >= 0", df_silver, F.col("tributos") >= 0)                 # nova
dq_check(t, "valor_total >= 0", df_silver, F.col("valor_total") >= 0)           # nova
dq_check(t, "valor_total = valor_energia + tributos", df_silver,
         F.col("valor_total") == F.col("valor_energia") + F.col("tributos"))
dq_check(t, "valores de origem com no máximo 2 casas decimais (o cast para decimal não arredondou)", df_final,
         ~(F.col("valor_energia_txt").rlike(r"\.[0-9]{3,}$")
           | F.col("tributos_txt").rlike(r"\.[0-9]{3,}$")
           | F.col("valor_total_txt").rlike(r"\.[0-9]{3,}$")))                  # nova

# Datas
dq_check(t, "emissão <= vencimento", df_silver, F.col("data_emissao") <= F.col("data_vencimento"))
dq_check(t, "vencimento = emissão + 12 dias (confirma a ordem dia/mês)", df_silver,
         F.datediff("data_vencimento", "data_emissao") == 12)

# Pagamento e flag em_aberto
dq_check(t, "em_aberto = data_pagamento nula", df_silver, F.col("em_aberto") == F.col("data_pagamento").isNull())
dq_check(t, "pagamento >= emissão (quando há pagamento)", df_silver,
         F.col("data_pagamento").isNull() | (F.col("data_pagamento") >= F.col("data_emissao")))
dq_check(t, "nenhum pagamento na sentinela (ano 9999)", df_silver,
         F.col("data_pagamento").isNull() | (F.year("data_pagamento") < 9999))

# Integridade referencial com a Silver de leituras (uma fatura por leitura)
leit = spark.table(f"{silver_schema}.leituras").select("id_uc", "mes_referencia").distinct()
pares_leit = leit.withColumn("tem_leitura", F.lit(True))
pares_fat = df_silver.select("id_uc", "mes_referencia").distinct().withColumn("tem_fatura", F.lit(True))

fat_com_leitura = (df_silver.join(pares_leit, ["id_uc", "mes_referencia"], "left")
                   .withColumn("tem_leitura", F.coalesce("tem_leitura", F.lit(False))))
leit_com_fatura = (leit.join(pares_fat, ["id_uc", "mes_referencia"], "left")
                   .withColumn("tem_fatura", F.coalesce("tem_fatura", F.lit(False))))

dq_check(t, "toda fatura tem leitura no mesmo mês (silver.leituras)", fat_com_leitura, F.col("tem_leitura"))
dq_check(t, "toda leitura tem fatura no mesmo mês", leit_com_fatura, F.col("tem_fatura"))

[PASS] silver.faturas | total de linhas = 2.152.336 | 2.152.336 linhas (esperado 2.152.336)
[PASS] silver.faturas | id_fatura único | 0 chaves duplicadas de 2152336 linhas
[PASS] silver.faturas | (id_uc, mes_referencia) único | 0 chaves duplicadas de 2152336 linhas
[PASS] silver.faturas | id_fatura não nulo | 0/2152336 linhas falharam
[PASS] silver.faturas | id_uc não nulo | 0/2152336 linhas falharam
[PASS] silver.faturas | mes_referencia não nulo | 0/2152336 linhas falharam
[PASS] silver.faturas | data_emissao não nulo | 0/2152336 linhas falharam
[PASS] silver.faturas | data_vencimento não nulo | 0/2152336 linhas falharam
[PASS] silver.faturas | valor_energia não nulo | 0/2152336 linhas falharam
[PASS] silver.faturas | tributos não nulo | 0/2152336 linhas falharam
[PASS] silver.faturas | valor_total não nulo | 0/2152336 linhas falharam
[PASS] silver.faturas | bandeira_tarifaria não nulo | 0/2152336 linhas falharam
[PASS] silver.faturas | status não nulo | 0/2152336 linhas falharam
[PA

## 12. Registro das checagens em `silver.dq_log`

O resultado das checagens é gravado **sempre**, mesmo quando alguma falha: o registro de uma falha é útil para investigar.

A gravação usa o modo `append` (o mesmo da aula), então **cada execução acrescenta linhas** e a tabela acumula as checagens de todas as execuções. O que diferencia uma execução da outra é a coluna `checked_at`. Ao consultar o log, filtre por `table_name` e pela execução mais recente, senão as mesmas checagens aparecem repetidas. A célula abaixo mostra só a execução atual.

In [0]:
# Persistência do dq_log (modo append, igual à aula); sempre grava, inclusive quando há falha
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{silver_schema}.dq_log")

# mostra só a execução atual (as últimas len(dq_results) linhas desta tabela)
display(
    spark.table(f"{silver_schema}.dq_log")
    .filter(F.col("table_name") == "silver.faturas")
    .orderBy(F.col("checked_at").desc())
    .limit(len(dq_results))
)

table_name,check_name,total_rows,failed_rows,passed,checked_at
silver.faturas,toda leitura tem fatura no mesmo mês,2152336,0,true,2026-10-22T19:41:31.885Z
silver.faturas,toda fatura tem leitura no mesmo mês (silver.leituras),2152336,0,true,2026-10-22T19:41:28.815Z
silver.faturas,nenhum pagamento na sentinela (ano 9999),2152336,0,true,2026-10-22T19:41:24.774Z
silver.faturas,pagamento >= emissão (quando há pagamento),2152336,0,true,2026-10-22T19:41:22.256Z
silver.faturas,em_aberto = data_pagamento nula,2152336,0,true,2026-10-22T19:41:18.188Z
silver.faturas,vencimento = emissão + 12 dias (confirma a ordem dia/mês),2152336,0,true,2026-10-22T19:41:15.866Z
silver.faturas,emissão <= vencimento,2152336,0,true,2026-10-22T19:41:12.539Z
silver.faturas,valores de origem com no máximo 2 casas decimais (o cast para decimal não arredondou),2152336,0,true,2026-10-22T19:41:09.034Z
silver.faturas,valor_total = valor_energia + tributos,2152336,0,true,2026-10-22T19:41:05.826Z
silver.faturas,valor_total >= 0,2152336,0,true,2026-10-22T19:41:03.103Z


## 13. Gravação da Silver, só se todas as checagens passarem

A Gold lê a Silver. Por isso, se qualquer checagem falhar, a tabela **não é gravada**: o notebook imprime quais checagens falharam e é interrompido com erro (num Workflow, isso marca a execução como falha e impede que dados com problema sigam para as camadas seguintes). Se `silver.faturas` já existia de uma execução anterior, ela permanece como estava.

In [0]:
# Gravação da Silver: só acontece se todas as checagens passaram
falhas = [r for r in dq_results if not r.passed]

if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_silver} NÃO foi gravada:")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_silver} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver)
)
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_silver} gravada: "
      f"{n_br(spark.table(tabela_silver).count())} linhas")

Todas as 29 checagens passaram. v_energy.silver.faturas gravada: 2.152.336 linhas


## 14. Bronze × Silver, com as mesmas linhas

Para cada tratamento, os `id_fatura` são escolhidos e depois mostrados nas duas tabelas, então dá para ver o **texto original** da Bronze (`R$ 100,64`, `9999-12-31`, `05-19-2023`) ao lado do valor tratado na Silver. Nos casos de cabeçalho repetido e de órfã, a Bronze mostra as linhas e a Silver não mostra nenhuma.

In [0]:
# Bronze × Silver: os MESMOS id_fatura nas duas tabelas, um grupo de exemplos por tratamento
silver_t = spark.table(tabela_silver)
ids_silver = silver_t.select("id_fatura")
colunas_bronze = ["id_fatura", "id_uc", "mes_referencia", "data_emissao", "valor_energia", "tributos",
                  "valor_total", "data_vencimento", "data_pagamento", "status"]

def comparar(titulo, ids_df, n=3):
    ids = [r["id_fatura"] for r in ids_df.select("id_fatura").limit(n).collect()]
    print(f"=== {titulo} | ids: {ids}")
    print("Bronze (texto original):")
    display(bronze.filter(F.col("id_fatura").isin(ids)).select(*colunas_bronze).orderBy("id_fatura"))
    print("Silver (tratado):")
    display(silver_t.filter(F.col("id_fatura").isin(ids)).orderBy("id_fatura"))

# 1. Cabeçalho repetido: existe na Bronze, não existe na Silver
comparar("Cabeçalho repetido", bronze.filter(e_cabecalho), n=2)

# 2. Órfãs: existem na Bronze, não existem na Silver
comparar("Órfãs (UC inexistente)", df_orfas, n=2)

# 3. valor_total como moeda: um exemplo de cada formato (R$ 9,9 e R$ 9.9,9)
ids_moeda = (
    bronze.join(ids_silver, "id_fatura", "left_semi")
    .filter(F.col("valor_total").startswith("R$"))
    .withColumn("padrao_valor", F.regexp_replace(F.regexp_replace("valor_total", "[0-9]", "9"), "9+", "9"))
    .dropDuplicates(["padrao_valor"])
)
comparar("valor_total como moeda (um de cada formato)", ids_moeda, n=2)

# 4. data_emissao: um exemplo de cada formato fora do ISO
ids_datas = (
    bronze.join(ids_silver, "id_fatura", "left_semi")
    .withColumn("padrao", F.regexp_replace("data_emissao", "[0-9]", "9"))
    .filter("padrao != '9999-99-99'")
    .dropDuplicates(["padrao"])
)
comparar("data_emissao fora do ISO (um de cada formato)", ids_datas, n=3)

# 5. data_pagamento ausente: um exemplo de nulo real e um de sentinela (9999-12-31)
ids_pagamento = (
    bronze.join(ids_silver, "id_fatura", "left_semi")
    .filter(F.col("data_pagamento").isNull() | (F.col("data_pagamento") == SENTINELA))
    .withColumn("forma", F.when(F.col("data_pagamento").isNull(), "nulo_real").otherwise("sentinela"))
    .dropDuplicates(["forma"])
)
comparar("data_pagamento ausente (nulo real e sentinela)", ids_pagamento, n=2)

=== Cabeçalho repetido | ids: ['id_fatura', 'id_fatura']
Bronze (texto original):


id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,valor_total,data_vencimento,data_pagamento,status
id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,valor_total,data_vencimento,data_pagamento,status
id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,valor_total,data_vencimento,data_pagamento,status


Silver (tratado):


id_fatura,id_uc,mes_referencia,data_emissao,data_vencimento,data_pagamento,valor_energia,tributos,valor_total,bandeira_tarifaria,status,em_aberto,source_file


=== Órfãs (UC inexistente) | ids: ['FTX0000021', 'FTX0000024']
Bronze (texto original):


...

id_fatura,id_uc,mes_referencia,data_emissao,data_vencimento,data_pagamento,valor_energia,tributos,valor_total,bandeira_tarifaria,status,em_aberto,source_file
FT000034426,UC100719,2023-04,2023-04-25,2023-05-07,2023-05-05,101.85,33.95,135.80,Verde,PAGA,false,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv
FT000057574,UC101202,2023-05,2023-05-13,2023-05-25,2023-06-05,73.19,24.40,97.59,Verde,PAGA,false,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv
FT000065422,UC101365,2025-09,2025-09-06,2025-09-18,2025-09-19,2701.62,900.54,3602.16,Vermelha P2,PAGA,false,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv


=== data_pagamento ausente (nulo real e sentinela) | ids: ['FT000010322', 'FT000113494']
Bronze (texto original):


id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,valor_total,data_vencimento,data_pagamento,status
FT000010322,UC100215,2023-03,2023-03-27,61.96,20.65,82.61,2023-04-08,null,EM ABERTO
FT000113494,UC102371,2022-08,2022-08-09,27.79,9.26,37.05,2022-08-21,9999-12-31,EM ABERTO


Silver (tratado):


id_fatura,id_uc,mes_referencia,data_emissao,data_vencimento,data_pagamento,valor_energia,tributos,valor_total,bandeira_tarifaria,status,em_aberto,source_file
FT000010322,UC100215,2023-03,2023-03-27,2023-04-08,null,61.96,20.65,82.61,Verde,EM ABERTO,true,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv
FT000113494,UC102371,2022-08,2022-08-09,2022-08-21,null,27.79,9.26,37.05,Verde,EM ABERTO,true,dbfs:/Volumes/v_energy/bronze/landing/faturas.csv
